In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
df=pd.read_csv('.././Imdb_Sentiment_Analysis/datasets/IMDB Dataset.csv')

In [20]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [154]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [156]:
map_sentiment={'positive':1,'negative':0}

In [160]:
df['sentiment']=df['sentiment'].map(map_sentiment)

In [22]:
df.shape

(50000, 2)

In [24]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

### Text Preprossing
#### Convert Text into lower and remove html tags,punctuations

In [29]:
df['review']=df['review'].str.lower()

In [35]:
import re
def remove_html(review):
    return re.sub(r'<.*?>', '', review)

In [39]:
df['review']=df['review'].apply(remove_html)

In [58]:
import string

def remove_punctuation(text):
    # Create a translation table to remove punctuation
    translation_table = str.maketrans('', '', string.punctuation)
    
    # Use the translate method to remove punctuation
    return text.translate(translation_table)


In [62]:
df['review']=df['review'].apply(remove_punctuation)

In [60]:
df['review']

0        one of the other reviewers has mentioned that ...
1        a wonderful little production. the filming tec...
2        i thought this was a wonderful way to spend ti...
3        basically there's a family where a little boy ...
4        petter mattei's "love in the time of money" is...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot, bad dialogue, bad acting, idiotic di...
49997    i am a catholic taught in parochial elementary...
49998    i'm going to have to disagree with the previou...
49999    no one expects the star trek movies to be high...
Name: review, Length: 50000, dtype: object

### Tokenization

In [55]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer

In [67]:
tokenizer=Tokenizer(oov_token='<nothing>')

In [69]:
tokenizer.fit_on_texts(df['review'])

In [73]:
len(tokenizer.word_index)

223805

In [ ]:
tokenizer.word_index

In [83]:
sequences=tokenizer.texts_to_sequences(list(df['review']))

In [97]:
max_len=max([len(item) for item in sequences])

## Padding Sequences

In [91]:
from tensorflow.keras.utils import pad_sequences

In [101]:
padded_input_sequences=pad_sequences(sequences,maxlen=max_len,padding='pre')

In [115]:
padded_input_sequences.shape

(50000, 2450)

In [117]:
padded_input_sequences

array([[    0,     0,     0, ...,   122,  3988,   502],
       [    0,     0,     0, ...,  1890,    73,   223],
       [    0,     0,     0, ...,    64,    15,   331],
       ...,
       [    0,     0,     0, ..., 22953,     3,  5957],
       [    0,     0,     0, ...,    68,   708,    42],
       [    0,     0,     0, ...,   840,    11,    17]])

## Spliting Data into training and testing

In [120]:
from sklearn.model_selection import train_test_split

In [178]:
X_train,X_test,y_train,y_test=train_test_split(padded_input_sequences,df.sentiment,test_size=0.2,random_state=42)

In [180]:
X_train.shape

(40000, 2450)

## Model Building

In [139]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,LSTM,GRU,SimpleRNN,Dropout,BatchNormalization,Embedding

In [182]:
model=Sequential()
model.add(Embedding(223805,250,input_length=2450))
model.add(LSTM(150,return_sequences=True))
model.add(LSTM(150))
model.add(Dense(1,activation='sigmoid'))

C:\Users\Admin\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [184]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_2 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [187]:
model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

In [189]:
model.fit(X_train,y_train,epochs=10,validation_data=(X_test,y_test))

Epoch 1/10
  88/1250 ━━━━━━━━━━━━━━━━━━━━ 6:48:39 21s/step - accuracy: 0.5854 - loss: 0.6563

KeyboardInterrupt: 

In [176]:
y_train

39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64

In [174]:
X_train.shape

(40000, 2450)